In [1]:
from pathlib import Path
from subprocess import run
import copykitten
import secrets
from base64 import urlsafe_b64encode

# Creating Keys for `SSH`

You have probably heard that SSH keys are a secure way to connect to a remote server. If you've ever used them, you know that they can also be very convenient. 

Secure and convenient? What's not to like? 

This is a downside. SSH is secure because it uses asymmetric cryptography. Asymmetric cryptography is difficult for people to understand. Famously, it leaves the user with the challenging task of managing secrets that are impossible to memorize. SSH was introduced in 1999. Its code is reliable because it has been so used so extensively. It is available on every operating system you are likely to use. But because of its age and its commitment to supporting so many platforms, it does not cater to users of  macOS or Windows. 

In practice, users on macOS and Windows tend to run a series of commands they don't understand. One of the most influential sources of advice about what commands to run is github.com. 

The goal in this notebook and the ones that follow is not to set up SSH as quickly as possible. It is to help you understand the choices you make. 

To help you understand, these notebooks make a clear distinction between two different activities:

- Authenticating with SSH

- Hardening SSH 

This notebook will help you implement the basic steps required to get SSH authentication working for someone who uses a workstation that runs macOS or Windows and who wants to connect with a remote server. 

The second notebooks, which differ for the to operating systems, describes options for isolating SSH from malware that could infect the user's workstation and run with the users privileges. The notebook for macOS shows that there is a way to stop malware from using SSH to connect to other machines that is relatively convenient to use. The companion notebook shows why this solution is impossible on Windows and explains the limited options for hardening SSH on Windows. 

## I. Solving the Authentication Problem 

In what follows, think of a key as a string that is like a password but too long to memorize. Here is what a key looks like in base64:

In [2]:
m = secrets.randbits(256)
c = urlsafe_b64encode(m.to_bytes(32))
c

b'13lq4rMvUKdD4Ic_QorQalN585XhnfdAAnqdBu95oac='

The "=" character at the end is a padding character. All other positions can take on 64 different values. 

When a client and a server connect using symmetric cryptography, both machines have access to the same secret. 

When the connect using asymmetric cryptography, each machine has a key pair made up of a secret key and a public key. 

### How the client authenticates
Authenticate means prove who you are. The client is identified by her public key. She authenticates by proving that she is the person who controls the secret that is mathematically connected to her public key. 

During a registration process, the client sets up an an account on the server that includes a copy of her public key. To authenticate, she sends her public key, says that she is the person who owns the account that is identified by that public key, and then proves that she does in fact control the secret that is paired with the public key. The useful feature of this system is that the client never has to tell the server her secret. She never has to send the secret over the wire. As the name suggests, her public key is public. Anyone can look it up. The client can authenticate with the server even though the server does not hold any secret information about the client. 

### How the server authenticates
There is, of course, a symmetric process. If the server has arranged for its public key to be given to the client, when the client and server are setting up a connection, the client can ask the server to prove that it controls the secret that is paired with the advertised public key. If the client always connects to the server that controls a specific secret key, she knows that she each time she connects, it is with a server controlled by the same organization. 



### Proof of knowledge

It takes some fascinating math explain how an entity can prove that it knows a secret without disclosing the secret. This math is covered in the notebook called challenge_response.ipynb. You will want to read it later. For now, the focus is on something known as the &ldquo;public key distribution problem.&rdquo; How, in practice, does a client get her public key stored on the server and how does the server tell the client it's public. 

They can't set up a secure SSH until they have distributed the public keys. They need some other secure communication channel that they can use to distribute the keys. To illustrate how this works in practice, it helps to focus on a specific case. How can you as the client set up an SSH connection to github.com. 

### The Four Steps That Solve the Authentication Problem
1. Create your key-pair
2. Tell the operating system to use that keypair when you want to connect to github.com
3. Log into github.com using http authentication; save your public key with Github.
4. Look at documenentation that Github controls for its public key; save a copy on your workstation. 

### Step 1: Create your key pair
The first step is to create a key pair. The public key is linked to the secret key via algorithms that are specified in a protocol. The discussion here will assume that the client and the server are using the most widely supported protocol, Ed25519. 

For SSH authentication, you need a key pair. To locate it after you create it, you must give it a name. 

This command provides a default name `ed25519` that specifies the protocol, but it is just name. You can change it to any other string before you run the code in the next cell. 

In [3]:
key_name = "id_ed2551_3"

Using this name, SSH will create two new files in a folder named `.ssh` that is in your user home directory. If it doesn't exist, SSH will create it for you. Because its name starts with a period, it is a hidden folder. You have to do some work to be able to the files that it contains. One of the files that SSH will put in this folder will have file extension. It is the file that contains your secret key. The other will have the file extension ".pub". It contains your public key. 

The names are an important reminder. You can reveal your public key to anyone, including github. Someone who knows your public key cannot impersonate you. But if someone learns your secret key, authentication is completely broken. They can impersonate you. 

The code in next cell checks whether you already have a key pair that uses the value you selected for `key_name`.

In [4]:
ssh = Path.home() / ".ssh"
secret_key_file = ssh / key_name
public_key_file = ssh / (key_name + ".pub")

if secret_key_file.exists() or public_key_file.exists():
    print("Try a different value for key_name.")
    print(f"You already have a key with the name {key_name}.")
else:
    print("This name is ok.")

Try a different value for key_name.
You already have a key with the name id_ed2551_3.


If you already have a key that uses `key_name`, you can delete the existing keys if you are sure that you don't need them; for example, if you are working through this notebook a second time. If you are sure, here is a some Python that will do this for you if you un-comment it.  

In [5]:
# os.chdir(ssh)
# if secret_key_file.exists():
#     secret_key_file.unlink()
# if public_key_file.exists():
#     public_key_file.unlink()

If you aren't sure about whether you are using these keys, the safe option is to go back and pick a different value for `key_name`. 

The next cell runs a command that expects a value for the variable `passphrase`. It is hard-coded here as the empty string. This means that you will create a secret ssh key that is not protected by a passphrase. 

The approach followed here is to get SSH authentication working first. Then in a second notebook, hardening_ssh.ipynb, you will have a chance to evaluate various options for hardening SSH. Adding a passphrase will be one of those options. 

The code in the next cell will create a `.ssh` directory if it does not exist. Then it will save two files, one with your secret key, the other with your public key. To create the secret key, the `ssh-keygen` function in effect picks a random string of 256 bits. It then uses mathematical functions that are specified by the ed25519 protocol to calculate the public key that is implied by your secret key. 


In [6]:
def create_keypair(key_dir: Path, key_name: str) -> None:
    """A function that creates an ed25519 keypair and saves both keys 
    in the directory key_dir with the stem specified by key_name. The 
    secret key will not be protected by a passphrase. 
    """
    passphrase = ""
    secret_key_path = key_dir / key_name

    if secret_key_file.exists():
        print("Change the key_name and try again")
        print(f"The file {str(secret_key_path)} already exists.")
    else:
        cp = run(
                    [
                        "ssh-keygen", 
                        "-f", 
                        str(secret_key_file), 
                        "-N", 
                        passphrase,
                        "-C",
                        key_name,
                        "-q",
                    ]
                )
        if cp.returncode == 0:
            print(f"Keys were created in {str(key_dir)}.")
    return 



ssh.mkdir(exist_ok=True)
create_keypair(ssh, key_name)

Change the key_name and try again
The file /Users/pr/.ssh/id_ed2551_3 already exists.


If your keys were created, the function will say so. But it never hurts to check: 

In [7]:
print(f"{ssh / key_name} exists? {(ssh / key_name).exists()}")
print(f"{ssh / (key_name + ".pub")} exists? {(ssh / key_name).exists()}")

/Users/pr/.ssh/id_ed2551_3 exists? True
/Users/pr/.ssh/id_ed2551_3.pub exists? True


### Step 2: Tell Git which key pair to use for Github 

If you are in a folder that is connected to a repository on Github and you give the command `git push` or `git pull`, git will authenticate with Github and try to push your content to the server or pull the content that is on the server. The information that is saved in this folder specifies the particular repo that you want to use at the domain github.com, but git has no way to know which keypair it should use to authenticate with Github. Because you might have several, you need to tell git which keypair to use. 

The simplest way to do that is to have a file called `config` in your `.ssh` directory and to put an entry in that file that looks like this: 

In [8]:
print(f"Host github.com\n  IdentityFile {ssh / key_name}")

Host github.com
  IdentityFile /Users/pr/.ssh/id_ed2551_3


If you don't have a `config` file, the function in the next cell will create one and add this entry. 

If you already have a `config` file, the function will make a backup called `config.back`. Then  
- if you don't have an entry for Github, it will add one.
- if you have a different entry for Github, it will replace it with this one.


In [9]:
def update_config(key_name: str):
    """Update the config file in the .ssh folder to include 
    an entry for github using the new key specified by key_name.
    """
    config = (Path.home() / ".ssh/config")
    new_entry = ["host github.com", f"  IdentityFile {ssh / key_name}", ""]

    if not config.exists():
        # config = new_entry
        config.write_text("\n".join(new_entry))

    else:
        (config.parent / "config.back").write_text(config.read_text())
        new_cfg = []
        for line in config.read_text().splitlines():
            new_cfg.append(line.rstrip())
        if "host github.com" in new_cfg:
            start_github_entry = new_cfg.index("host github.com")
            try:
                start_next_entry = new_cfg.index("", start_github_entry) + 1
            except ValueError:
                start_next_entry = len(new_cfg)
            new_cfg[start_github_entry:start_next_entry] = new_entry

        else:
            new_cfg = new_cfg.extend(new_entry) 


        if not new_cfg[-1] == "":
            new_cfg.append("")

    s = "\n".join(new_cfg) 
    m = config.write_text(s)

    print("You now have a config file with this entry for github.com:")
    print(f"\nHost github.com\n  IdentityFile {ssh / key_name}")
    return


update_config(key_name)

You now have a config file with this entry for github.com:

Host github.com
  IdentityFile /Users/pr/.ssh/id_ed2551_3


### Step 3: Save your public key with Github

To save your public key with Github, you need to log in to its website using standard HTTPS authentication. 

Your public key is stored in this file:

In [ ]:
print(ssh / (key_name + ".pub"))

Once you are logged in, you need to copy the contents from that file and paste them into a window on the Github website. 

To get ready, open a browser in another window on your computer. Log into Github.com. If you don't already have an account with Github, you'll have to register. Once you are registered, use your browser to login. 

After you login, you land on the Home page. In the upper right corner, you'll see an icon that represents you. Click on it and scan the list that opens below. About 2/3rds of way down, you'll find the link `Settings`. After you click on that link, you'll land on the Settings page. 

On the Settings page, look down the list on the left side for the entry "SSH and GPG". Click on it. 

On the Keys page where you land, there is a green button in the upper right that says "New SSH key". Locate it, but don't click it yet. Instead, come back to this notebook and run the code in the next cell. It uses the copy_kitten library to put your public key onto the system clipboard. This makes it easy for you to do the required paste on Github. By relying on code, you are sure not to make the common mistake of copying and pasting the secret key onto Github.

In [ ]:
def publickey_to_clipboard(key_name):
    """Use copykitten.copy() to put the public key on the clipboard"""
    p = Path.home() / (".ssh/" + key_name + ".pub")
    s = p.read_text()
    copykitten.copy(s)
    return


publickey_to_clipboard(key_name)

After you run this cell, go back to your browser. On Github's Keys page, click on "New SSH key". You'll land on a new key page with:

- a title window
- a type selector
- a key window 

Click in the key window and use the paste keyboard-shortcut (CMD v on macOS or CTRL v on Windows). Your public key will be pasted into the window. 

All that's left is to confirm by clicking on the green "Add Key" button below the key window. 

You don't need to supply a name for the key. Github will read it from the comment at the end of your key. 

Leave the type selector in its default state, "Authentication Key." 

## Step 4: Get Github's public key 

At this point, you should be able to authenticate with the server. You have the local copy of your secret key. The server has your public key. 

But how do you know that you are connecting to the right server? 

The risk that you connect to the wrong server is low but it isn't zero. When you tell SSH to make a connection with github.com, SSH relies on the DNS system to find the IP address for github.com and then connects using that IP address. There are known attacks that can trick the DNS system into returning an IP address that an attacker specifies. When you tried to log into Github.com, this same type of attack may have tricked your browser into using the same incorrect IP address. There are other protections for the HTTPS protocol that your browser relies on. They would probably have caught the problem for the https connection, but you might still end up with an SSH connection to a server that a bad actor controls. The most secure way to use SSH is to make sure that you have a copy of the public key that corresponds to the secret key that Github uses. With that public key, you can ask the server you are connected to to prove that it has access to the secret key that is paired with Github's published public key. 

In practice, the risk that SSH connects to the wrong server is low. Almost everyone ignores it when they sign into a server the first time. By default, they follow a strategy known as "trust on first use." The first time they make an SSH connection to a server, they trust that they are connecting to the right one. When they connect, the server sends them its public key, which is saved on their computer. On all subsequent connections, they can confirm that they are connecting to the same server that they connected to before. Or to be more specific, they are connecting to a server that has access to the same secret key that the the server had the first time they connected. 

The only indication about what is going on with "trust on first use" is a message that SSH shows the first time you connect to a server. This next lines are from a terminal session in which I asked ssh to make a connection to git@github.com. 

```
$ ssh -T git@github.com
```

Here is the message I got back: 

```
The authenticity of host 'github.com (140.82.113.4)' can't be established.
ED25519 key fingerprint is: SHA256:+DiY3wvvV6TuJJhbpZisF/zLDA0zPMSvHdkr4UvCOqU
This key is not known by any other names.
Are you sure you want to continue connecting (yes/no/[fingerprint])? 
```

What most people do is answer yes and &ldquo;trust on first use.&rdquo; If I had responded by typing &lsquo;yes&rsquo; the connection would have gone through. SSH would have received the public key that github.com uses for the Ed25519 protocol and saved it in a file called `known_hosts` in my `.ssh` directory. 

But to drive home the principle that public keys need to be communicated through an independent channel to make a secure connection, I'll walk you through a different approach. I looked up the public key in the Github documentation. (If you want to check for yourself, the URL is in the doc string for the function in the next cell.) Here is an image that shows some of the information that Github: 

<dir>
    <img src="./github_keys.png">
</dir>

The first entry (which is a technically a single line but in the image below it spills over onto two lines) has the public key that is paired with the secret key that Github.com uses for SSH connections that rely on the Ed25519 protocol.

The code in the next cell creates a `known_hosts` file if you don't have one and makes sure that you have Github's Ed25519 public key in that file. 

In [10]:
def check_known_hosts(s: str = ""):
    """Checks the known hosts file for a line that contains the string s. 
    If no string is supplied, the function uses the ed25519 entry
    for github.com. On Mar 14, 2026, Paul Romer copied this entry from 
    the github.com documentation at this url:
    url = "https://docs.github.com/en/authentication/"
    url += "keeping-your-account-and-data-secure/"
    url += "githubs-ssh-key-fingerprints"
    """
    if not s:
        s = "github.com ssh-ed25519 "
        s += "AAAAC3NzaC1lZDI1NTE5AAAAIOMqqnkVzrm"
        s += "0SdG6UOoqKLsabgH5C9okWi0dh2l9GKJl"

    found: bool = False
    known_hosts = Path.home() / ".ssh/known_hosts"
    known_hosts_list = known_hosts.read_text().splitlines()
    if known_hosts.exists():
        for line in known_hosts_list:
            if s in line:
                found = True
                break
    else:
        known_hosts_list = []
    if found:
        print("You already have an entry for Github in known_hosts.")
    if not found:
        known_hosts_list.append(s)
        m = known_hosts.write_text("\n".join(known_hosts_list))
        if m > 0:
            print("You now have an entry for Github in your known_hosts file.")
    return


check_known_hosts()

You already have an entry for Github in known_hosts.


## Ready to authenticate

At this point, you and the server are in the same position. 

- You have a secret key and the server knows your public key
- The server has a secret key and you know the server's public key

You and the github server can now authenticate securely with each other. 

To confirm, run the test in the next cell. It will make an SSH connection to github and reply with your github ID. 

In [11]:
def test_connection():
    """Tests ssh authentication to Github. If you authenticate,  
    you you should get a congratulatory message that contains your 
    github ID.
    """
    cmd_lst = ["ssh", "-T", "ssh://git@github.com:"]
    cp = run(cmd_lst, capture_output=True, text=True)

    github_id = cp.stderr[:cp.stderr.find("!")]
    github_id = github_id[github_id.find(" ") + 1:]

    if "You've successfully authenticated" in cp.stderr:
        print()
        print(f"    Congratulations {github_id}!")
        print()
        print("    You successfully authenticated with github.com")
        print()


test_connection()


    Congratulations paulmromer!

    You successfully authenticated with github.com



If you don't see a message congratulating you, check all the steps outlined above, including the ones that require that you log in to your github account and paste your public key there. 

If you created a new key pair, be sure that you also update the value of the public key that you paste in at Github.  

## What next? 

At this point, you should be able to get updates of the content in any of the channels in Gennaker. If you want to test this, you can open a terminal. For this channel, the terminal should open inside `~/Gennaker/Channels/DSD/Files`. If you enter `cd ..` you will  move up one level to `~/Gennaker/Channels/DSD`. Next, enter `cd Pubs` to change to the `Pubs` directory. There you can open a terminal and run the command `git status` to see the current status for the files in `Pubs`. If you want, you can run `git pull`. If there is any new content on the server, this command will bring it down. But there is no need for you to do this. Each time you start Gennaker, it will run the `git pull` command. So after each restart, you'll be up to date. 

Now that SSH authentication is working, there are two additional steps: 

- There is (or there will soon be) a notebook called `challenge_response.ipynb` in the `Pubs` directory of `DSD`. It explains how a key pair lets someone prove that they have a secret key without revealing it. 

- There is (or will soon be) a notebook in the same directory called `file_permissions.ipynb` and another one called `hardening_ssh.ipynb`. They show how you can protect your SSH setup from misuse by malware.

If there were zero risk that malware might run on your computer, your current setup would be fine. Realistically, there is a real risk that you will end up with malware on your computer at some point. The `hardening_ssh` notebook explains the trade-offs you face when you try to limit the damage that malware could do if it got onto your computer. In particular, it considers the trade-offs associated with the use of passphrases that encrypt an SSH secret key. 

You shouldn't follow anyone's advice about what to do until you understand the security-convenience trade-offs you face. 

